# Phase 1B - Column Analysis, Parsing & Cleanup

## Objective

Analyze Phase 1 output, parse structured data from text fields, and prepare data for feature engineering.

---

## Phase 1 Recap

**Input**: 3,372,330 raw records (LogFile + UsnJrnl)
**Output**: 154,550 filtered records with smart union strategy
**Timestomped events**: 252 events labeled (100% capture rate)
**Columns**: 27 columns total

---

## 1. Setup & Load Data

In [65]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")

✓ Libraries imported successfully
Pandas version: 2.3.2


In [66]:
# Define paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
INPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 1 - Data Cleaning'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 1B - Column Cleanup'

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("📂 Directory Configuration:")
print(f"  Input: {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")

📂 Directory Configuration:
  Input: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1 - Data Cleaning
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1B - Column Cleanup


In [67]:
# Load master dataset (ONLY master dataset, no individual case files)
print("Loading master dataset...")
df = pd.read_csv(INPUT_DIR / 'all_cases_combined.csv', encoding='utf-8-sig')

print(f"\n✓ Loaded dataset:")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Timestomped events: {(df['is_timestomped'] == 1).sum()}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print(f"  Cases: {sorted(df['case_id'].unique().tolist())}")

Loading master dataset...

✓ Loaded dataset:
  Records: 154,550
  Columns: 27
  Timestomped events: 252
  Memory usage: 174.81 MB
  Cases: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


---
## 2. Overall Data Quality Assessment

In [68]:
print("\n" + "=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

# Show column list
print("\n📋 Current Columns (27):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")


DATASET OVERVIEW

📋 Current Columns (27):
   1. lf_lsn
   2. eventtime
   3. lf_event
   4. lf_detail
   5. filename
   6. filepath
   7. lf_creation_time
   8. lf_modified_time
   9. lf_mft_modified_time
  10. lf_accessed_time
  11. lf_redo
  12. lf_target_vcn
  13. lf_cluster_index
  14. case_id
  15. eventtime_dt
  16. merge_key
  17. usn_usn
  18. usn_event_info
  19. usn_source_info
  20. usn_file_attribute
  21. usn_carving_flag
  22. usn_file_reference_number
  23. usn_parent_file_reference_number
  24. source
  25. time_diff_seconds
  26. is_tunneling
  27. is_timestomped


In [69]:
# Missing data summary
print("\n" + "=" * 80)
print("MISSING DATA ANALYSIS")
print("=" * 80)

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2),
    'Non_Missing_Count': df.notnull().sum()
})

missing_data = missing_data.sort_values('Missing_Percent', ascending=False).reset_index(drop=True)

print("\n📊 Missing Data Summary (sorted by missing %):")
print(missing_data.to_string(index=False))


MISSING DATA ANALYSIS

📊 Missing Data Summary (sorted by missing %):
                          Column  Missing_Count  Missing_Percent  Non_Missing_Count
                lf_accessed_time         154550           100.00                  0
                usn_carving_flag         154550           100.00                  0
                lf_creation_time         154550           100.00                  0
                lf_modified_time         154550           100.00                  0
            lf_mft_modified_time         154550           100.00                  0
               time_diff_seconds         151686            98.15               2864
                          lf_lsn         151448            97.99               3102
                lf_cluster_index         151448            97.99               3102
                         lf_redo         151448            97.99               3102
                   lf_target_vcn         151448            97.99               3102
      

---
## 3. LogFile Event Type Analysis

**Question**: What types of LogFile events do we actually have?

In [70]:
print("\n" + "=" * 80)
print("LOGFILE EVENT TYPE ANALYSIS")
print("=" * 80)

# Filter to LogFile records
lf_records = df[df['source'].isin(['both', 'logfile_only'])].copy()
print(f"\n📊 LogFile records: {len(lf_records):,} out of {len(df):,} total")

# Check event types
print(f"\n📋 LogFile Event Type Distribution:")
event_counts = lf_records['lf_event'].value_counts()
print(event_counts)

# Calculate percentages
print(f"\n📊 Event Type Breakdown:")
for event, count in event_counts.items():
    pct = (count / len(lf_records) * 100)
    print(f"  {event:50s}: {count:5,} ({pct:5.1f}%)")


LOGFILE EVENT TYPE ANALYSIS

📊 LogFile records: 3,102 out of 154,550 total

📋 LogFile Event Type Distribution:
lf_event
Time Reversal Event                             2814
Time Reversal Event & Changing FileAttribute     288
Name: count, dtype: int64

📊 Event Type Breakdown:
  Time Reversal Event                               : 2,814 ( 90.7%)
  Time Reversal Event & Changing FileAttribute      :   288 (  9.3%)


### Key Finding: Only Time Reversal Events

**Conclusion**:
- ✅ We ONLY have "Time Reversal Event" and "Time Reversal Event & Changing FileAttribute"
- ✅ All LogFile records are explicit timestamp manipulation indicators

---
## 4. Single-Value Column Analysis

**Question**: Are there columns with only one unique value (no variance)?

In [71]:
print("\n" + "=" * 80)
print("SINGLE-VALUE COLUMN ANALYSIS")
print("=" * 80)

# Check unique value counts for all columns
unique_counts = pd.DataFrame({
    'Column': df.columns,
    'Unique_Values': [df[col].nunique() for col in df.columns],
    'Unique_Including_NaN': [df[col].nunique(dropna=False) for col in df.columns]
})

unique_counts = unique_counts.sort_values('Unique_Values')

print("\n📊 Unique Value Counts (sorted ascending):")
print(unique_counts.to_string(index=False))

# Highlight single-value columns
single_value = unique_counts[unique_counts['Unique_Values'] <= 1]
print(f"\n\n⚠️  Columns with ≤1 unique values (excluding NaN):")
if len(single_value) > 0:
    print(single_value.to_string(index=False))
else:
    print("  None")


SINGLE-VALUE COLUMN ANALYSIS

📊 Unique Value Counts (sorted ascending):
                          Column  Unique_Values  Unique_Including_NaN
                usn_carving_flag              0                     1
                lf_creation_time              0                     1
                lf_modified_time              0                     1
            lf_mft_modified_time              0                     1
                lf_accessed_time              0                     1
                 usn_source_info              1                     2
                         lf_redo              1                     2
                  is_timestomped              2                     2
               time_diff_seconds              2                     3
                        lf_event              2                     3
                    is_tunneling              2                     2
                          source              3                     3
                l

In [72]:
# Investigate single-value columns in detail
print("\n\n📋 Detailed Analysis of Single-Value Columns:")

# Check usn_source_info
print("\n1. usn_source_info:")
usn_records = df[df['source'].isin(['both', 'usnjrnl_only'])]
print(f"   Total UsnJrnl records: {len(usn_records):,}")
print(f"   Value distribution:")
print(df['usn_source_info'].value_counts(dropna=False))
print(f"   Conclusion: Only 'Normal' (no variance) → REMOVE")

# Check lf_redo
print("\n2. lf_redo:")
print(f"   Total LogFile records: {len(lf_records):,}")
print(f"   Value distribution:")
print(lf_records['lf_redo'].value_counts(dropna=False))
print(f"   Conclusion: Only 'Update Resident Value' (no variance) → REMOVE")

# Check usn_carving_flag
print("\n3. usn_carving_flag:")
print(f"   Total UsnJrnl records: {len(usn_records):,}")
print(f"   Value distribution:")
print(df['usn_carving_flag'].value_counts(dropna=False))
print(f"   Conclusion: 100% empty (no data) → REMOVE")



📋 Detailed Analysis of Single-Value Columns:

1. usn_source_info:
   Total UsnJrnl records: 154,312
   Value distribution:
usn_source_info
Normal    154312
NaN          238
Name: count, dtype: int64
   Conclusion: Only 'Normal' (no variance) → REMOVE

2. lf_redo:
   Total LogFile records: 3,102
   Value distribution:
lf_redo
Update Resident Value    3102
Name: count, dtype: int64
   Conclusion: Only 'Update Resident Value' (no variance) → REMOVE

3. usn_carving_flag:
   Total UsnJrnl records: 154,312
   Value distribution:
usn_carving_flag
NaN    154550
Name: count, dtype: int64
   Conclusion: 100% empty (no data) → REMOVE


### Key Finding: Three Useless Columns

**Columns to Remove:**
1. `usn_source_info` - 100% "Normal" (no variance, provides zero information for ML)
2. `lf_redo` - 100% "Update Resident Value" (no variance, all Time Reversal events have same operation)
3. `usn_carving_flag` - 100% empty (no data at all)

**Rationale**: Columns with no variance cannot be used as features in ML models.

---
## 5. LogFile Timestamp Columns Analysis

**Question**: Why are `lf_creation_time`, `lf_modified_time`, `lf_accessed_time`, `lf_mft_modified_time` empty?

In [73]:
print("\n" + "=" * 80)
print("LOGFILE TIMESTAMP COLUMNS ANALYSIS")
print("=" * 80)

timestamp_cols = ['lf_creation_time', 'lf_modified_time', 'lf_accessed_time', 'lf_mft_modified_time']

print(f"\n📋 Missing data in LogFile timestamp columns:")
for col in timestamp_cols:
    missing_count = lf_records[col].isnull().sum()
    missing_pct = (missing_count / len(lf_records) * 100)
    print(f"  {col:25s}: {missing_count:5,} / {len(lf_records):,} ({missing_pct:.1f}%)")

print("\n✓ Conclusion: ALL timestamp columns are empty for Time Reversal events")
print("  Reason: Timestamp details are stored in lf_detail text field instead")


LOGFILE TIMESTAMP COLUMNS ANALYSIS

📋 Missing data in LogFile timestamp columns:
  lf_creation_time         : 3,102 / 3,102 (100.0%)
  lf_modified_time         : 3,102 / 3,102 (100.0%)
  lf_accessed_time         : 3,102 / 3,102 (100.0%)
  lf_mft_modified_time     : 3,102 / 3,102 (100.0%)

✓ Conclusion: ALL timestamp columns are empty for Time Reversal events
  Reason: Timestamp details are stored in lf_detail text field instead


---
## 6. Parse `lf_detail` Column

**Objective**: Extract structured timestamp manipulation data from `lf_detail` text field

**Expected Format**:
```
CreationTime : 2023-12-31 01:15:49 -> 2023-12-25 16:22:59(same as wabimp.dll)
ModifiedTime : 2023-12-26 15:16:49 -> 2022-12-26 15:24:17(Zero in 100-nanoseconds)
```

In [74]:
print("\n" + "=" * 80)
print("LF_DETAIL COLUMN ANALYSIS")
print("=" * 80)

# Show sample lf_detail values
lf_with_detail = lf_records[lf_records['lf_detail'].notnull()]

print(f"\n📊 Records with lf_detail: {len(lf_with_detail):,} / {len(lf_records):,}")
print(f"\n📋 Sample lf_detail values (first 5):")
print("=" * 120)

for idx, row in lf_with_detail[['lf_event', 'lf_detail', 'filename']].head(5).iterrows():
    print(f"\nEvent: {row['lf_event']}")
    print(f"File: {row['filename']}")
    print(f"Detail: {row['lf_detail'][:200]}...")  # Truncate long details
    print("-" * 120)


LF_DETAIL COLUMN ANALYSIS

📊 Records with lf_detail: 3,102 / 3,102

📋 Sample lf_detail values (first 5):

Event: Time Reversal Event
File: 2bf78d23-dca6-45a7-a380-e1b17c1a5c30.tmp
Detail: CreationTime : 2023-12-26 00:04:10 -> 2022-12-16 17:14:37(same as Local State/ Local State~RF366ba.TMP)...
------------------------------------------------------------------------------------------------------------------------

Event: Time Reversal Event
File: 11a8ccd2-750d-4e27-adcc-6ad1a2db8ad7.tmp
Detail: CreationTime : 2023-12-26 00:04:37 -> 2022-12-16 17:32:19(same as Preferences~RF3bb04.TMP)...
------------------------------------------------------------------------------------------------------------------------

Event: Time Reversal Event
File: d863aa00-e831-4ca7-a2d8-11351677c750.tmp
Detail: CreationTime : 2023-12-26 00:04:37 -> 2022-12-16 17:32:19(same as Preferences/ Preferences~RF3bb04.TMP)...
-----------------------------------------------------------------------------------------------

In [75]:
# Analyze lf_detail patterns
print("\n\n📊 lf_detail Pattern Analysis:")

has_arrow = lf_with_detail['lf_detail'].str.contains('->', na=False)
has_zero_nano = lf_with_detail['lf_detail'].str.contains('Zero in 100-nanoseconds', na=False, case=False)
has_same_as = lf_with_detail['lf_detail'].str.contains('same as', na=False, case=False)

print(f"  Records with '->' (timestamp change): {has_arrow.sum():,} ({has_arrow.sum()/len(lf_with_detail)*100:.1f}%)")
print(f"  Records with 'Zero in 100-nanoseconds': {has_zero_nano.sum():,} ({has_zero_nano.sum()/len(lf_with_detail)*100:.1f}%)")
print(f"  Records with 'same as' (copied timestamp): {has_same_as.sum():,} ({has_same_as.sum()/len(lf_with_detail)*100:.1f}%)")

# Check which timestamp types are affected
has_creation = lf_with_detail['lf_detail'].str.contains('CreationTime', na=False, case=False)
has_modified = lf_with_detail['lf_detail'].str.contains('ModifiedTime', na=False, case=False)
has_accessed = lf_with_detail['lf_detail'].str.contains('AccessedTime', na=False, case=False)
has_mft_modified = lf_with_detail['lf_detail'].str.contains('MFTModifiedTime', na=False, case=False)

print(f"\n  Timestamp types manipulated:")
print(f"    CreationTime: {has_creation.sum():,} ({has_creation.sum()/len(lf_with_detail)*100:.1f}%)")
print(f"    ModifiedTime: {has_modified.sum():,} ({has_modified.sum()/len(lf_with_detail)*100:.1f}%)")
print(f"    AccessedTime: {has_accessed.sum():,} ({has_accessed.sum()/len(lf_with_detail)*100:.1f}%)")
print(f"    MFTModifiedTime: {has_mft_modified.sum():,} ({has_mft_modified.sum()/len(lf_with_detail)*100:.1f}%)")



📊 lf_detail Pattern Analysis:
  Records with '->' (timestamp change): 3,102 (100.0%)
  Records with 'Zero in 100-nanoseconds': 1,298 (41.8%)
  Records with 'same as' (copied timestamp): 152 (4.9%)

  Timestamp types manipulated:
    CreationTime: 473 (15.2%)
    ModifiedTime: 2,919 (94.1%)
    AccessedTime: 292 (9.4%)
    MFTModifiedTime: 1,469 (47.4%)


### 6.1 Create Parsing Function

In [76]:
def parse_lf_detail(detail_str):
    """
    Parse lf_detail text field to extract timestamp manipulation details.
    
    Input format examples:
    - "CreationTime : 2023-12-31 01:15:49 -> 2023-12-25 16:22:59(same as wabimp.dll)"
    - "ModifiedTime : 2023-12-26 15:16:49 -> 2022-12-26 15:24:17(Zero in 100-nanoseconds)"
    - "CreationTime : 2023-12-31 00:00:31 -> 2022-12-24 11:24:41"
    
    Returns:
        dict with keys:
        - lf_creation_time_before, lf_creation_time_after
        - lf_modified_time_before, lf_modified_time_after
        - lf_accessed_time_before, lf_accessed_time_after
        - lf_mft_modified_time_before, lf_mft_modified_time_after
        - zero_in_nanoseconds (bool)
        - copied_from_file (bool)
    """
    result = {
        'lf_creation_time_before': None,
        'lf_creation_time_after': None,
        'lf_modified_time_before': None,
        'lf_modified_time_after': None,
        'lf_accessed_time_before': None,
        'lf_accessed_time_after': None,
        'lf_mft_modified_time_before': None,
        'lf_mft_modified_time_after': None,
        'zero_in_nanoseconds': False,
        'copied_from_file': False
    }
    
    if pd.isna(detail_str):
        return result
    
    # Check for indicators
    result['zero_in_nanoseconds'] = 'Zero in 100-nanoseconds' in detail_str
    result['copied_from_file'] = 'same as' in detail_str
    
    # Parse timestamp changes
    # Pattern: "<TimestampType> : <before> -> <after>"
    timestamp_types = [
        ('CreationTime', 'lf_creation_time'),
        ('ModifiedTime', 'lf_modified_time'),
        ('AccessedTime', 'lf_accessed_time'),
        ('MFTModifiedTime', 'lf_mft_modified_time')
    ]
    
    for ts_name, col_prefix in timestamp_types:
        # Pattern: TimestampType : YYYY-MM-DD HH:MM:SS -> YYYY-MM-DD HH:MM:SS
        pattern = rf'{ts_name}\s*:\s*(\d{{4}}-\d{{2}}-\d{{2}}\s+\d{{2}}:\d{{2}}:\d{{2}})\s*->\s*(\d{{4}}-\d{{2}}-\d{{2}}\s+\d{{2}}:\d{{2}}:\d{{2}})'
        match = re.search(pattern, detail_str)
        
        if match:
            before = match.group(1)
            after = match.group(2)
            result[f'{col_prefix}_before'] = before
            result[f'{col_prefix}_after'] = after
    
    return result

print("✓ Parsing function created")

✓ Parsing function created


### 6.2 Test Parsing Function

In [77]:
print("\n" + "=" * 80)
print("TESTING LF_DETAIL PARSING FUNCTION")
print("=" * 80)

# Test on sample records
print("\n📋 Testing on 5 sample records:")

for idx, row in lf_with_detail.head(5).iterrows():
    print(f"\n--- Record {idx} ---")
    print(f"Original lf_detail:")
    print(f"  {row['lf_detail'][:150]}...")  # Truncate
    
    parsed = parse_lf_detail(row['lf_detail'])
    print(f"\nParsed results:")
    for key, value in parsed.items():
        if value is not None and value != False:
            print(f"  {key}: {value}")
    print("-" * 80)


TESTING LF_DETAIL PARSING FUNCTION

📋 Testing on 5 sample records:

--- Record 0 ---
Original lf_detail:
  CreationTime : 2023-12-26 00:04:10 -> 2022-12-16 17:14:37(same as Local State/ Local State~RF366ba.TMP)...

Parsed results:
  lf_creation_time_before: 2023-12-26 00:04:10
  lf_creation_time_after: 2022-12-16 17:14:37
  copied_from_file: True
--------------------------------------------------------------------------------

--- Record 1 ---
Original lf_detail:
  CreationTime : 2023-12-26 00:04:37 -> 2022-12-16 17:32:19(same as Preferences~RF3bb04.TMP)...

Parsed results:
  lf_creation_time_before: 2023-12-26 00:04:37
  lf_creation_time_after: 2022-12-16 17:32:19
  copied_from_file: True
--------------------------------------------------------------------------------

--- Record 2 ---
Original lf_detail:
  CreationTime : 2023-12-26 00:04:37 -> 2022-12-16 17:32:19(same as Preferences/ Preferences~RF3bb04.TMP)...

Parsed results:
  lf_creation_time_before: 2023-12-26 00:04:37
  lf_cre

### 6.3 Apply Parsing to Entire Dataset

In [78]:
print("\n" + "=" * 80)
print("APPLYING LF_DETAIL PARSING TO DATASET")
print("=" * 80)

print("\nParsing lf_detail for all records...")
print("This may take a moment...")

# Parse lf_detail
parsed_data = df['lf_detail'].apply(parse_lf_detail)
parsed_df = pd.DataFrame(parsed_data.tolist())

# Add parsed columns to main dataframe
df_parsed = pd.concat([df, parsed_df], axis=1)

print(f"\n✓ Parsing complete!")
print(f"  Original columns: {len(df.columns)}")
print(f"  New parsed columns: {len(parsed_df.columns)}")
print(f"  Total columns: {len(df_parsed.columns)}")

# Show parsing results summary
print(f"\n📊 Parsing Results Summary:")
new_cols = parsed_df.columns.tolist()
for col in new_cols:
    non_null = df_parsed[col].notnull().sum()
    if col.endswith('_before') or col.endswith('_after'):
        print(f"  {col:35s}: {non_null:5,} records populated")
    else:
        true_count = (df_parsed[col] == True).sum()
        print(f"  {col:35s}: {true_count:5,} records = True")


APPLYING LF_DETAIL PARSING TO DATASET

Parsing lf_detail for all records...
This may take a moment...

✓ Parsing complete!
  Original columns: 27
  New parsed columns: 10
  Total columns: 37

📊 Parsing Results Summary:
  lf_creation_time_before            :   473 records populated
  lf_creation_time_after             :   473 records populated
  lf_modified_time_before            : 2,919 records populated
  lf_modified_time_after             : 2,919 records populated
  lf_accessed_time_before            :   292 records populated
  lf_accessed_time_after             :   292 records populated
  lf_mft_modified_time_before        : 1,469 records populated
  lf_mft_modified_time_after         : 1,469 records populated
  zero_in_nanoseconds                : 1,298 records = True
  copied_from_file                   :   152 records = True


### Key Finding: Successful Timestamp Parsing

**New Columns Created:**
- ✅ `lf_creation_time_before`, `lf_creation_time_after`
- ✅ `lf_modified_time_before`, `lf_modified_time_after`
- ✅ `lf_accessed_time_before`, `lf_accessed_time_after`
- ✅ `lf_mft_modified_time_before`, `lf_mft_modified_time_after`
- ✅ `zero_in_nanoseconds` (boolean indicator)
- ✅ `copied_from_file` (boolean indicator)

**Old Empty Columns Can Now Be Removed:**
- ❌ `lf_creation_time` → Replaced by parsed before/after
- ❌ `lf_modified_time` → Replaced by parsed before/after
- ❌ `lf_accessed_time` → Replaced by parsed before/after
- ❌ `lf_mft_modified_time` → Replaced by parsed before/after

---
## 7. Calculate Timestamp Delta Features

In [79]:
print("\n" + "=" * 80)
print("CALCULATING TIMESTAMP DELTA FEATURES")
print("=" * 80)

# Convert timestamp strings to datetime
timestamp_pairs = [
    ('lf_creation_time_before', 'lf_creation_time_after', 'creation_time_delta_days', 'creation_time_changed_to_past'),
    ('lf_modified_time_before', 'lf_modified_time_after', 'modified_time_delta_days', 'modified_time_changed_to_past'),
    ('lf_accessed_time_before', 'lf_accessed_time_after', 'accessed_time_delta_days', 'accessed_time_changed_to_past'),
    ('lf_mft_modified_time_before', 'lf_mft_modified_time_after', 'mft_modified_time_delta_days', 'mft_modified_time_changed_to_past')
]

for before_col, after_col, delta_col, direction_col in timestamp_pairs:
    # Convert to datetime
    before_dt = pd.to_datetime(df_parsed[before_col], errors='coerce')
    after_dt = pd.to_datetime(df_parsed[after_col], errors='coerce')
    
    # Calculate delta (before - after, positive means changed to past)
    delta = (before_dt - after_dt).dt.total_seconds() / (24 * 3600)  # Convert to days
    df_parsed[delta_col] = delta
    
    # Boolean: changed to past (after < before)
    df_parsed[direction_col] = after_dt < before_dt
    
    # Show results
    non_null = df_parsed[delta_col].notnull().sum()
    if non_null > 0:
        print(f"\n{delta_col}:")
        print(f"  Records with delta: {non_null:,}")
        print(f"  Mean delta: {df_parsed[delta_col].mean():.2f} days")
        print(f"  Median delta: {df_parsed[delta_col].median():.2f} days")
        print(f"  Changed to past: {df_parsed[direction_col].sum():,} ({df_parsed[direction_col].sum()/non_null*100:.1f}%)")

print(f"\n✓ Delta features calculated successfully!")


CALCULATING TIMESTAMP DELTA FEATURES

creation_time_delta_days:
  Records with delta: 473
  Mean delta: 168.59 days
  Median delta: 66.37 days
  Changed to past: 473 (100.0%)

modified_time_delta_days:
  Records with delta: 2,919
  Mean delta: 2116.01 days
  Median delta: 66.37 days
  Changed to past: 2,919 (100.0%)

accessed_time_delta_days:
  Records with delta: 292
  Mean delta: 69.03 days
  Median delta: 66.37 days
  Changed to past: 292 (100.0%)

mft_modified_time_delta_days:
  Records with delta: 1,469
  Mean delta: 162.04 days
  Median delta: 5.38 days
  Changed to past: 1,469 (100.0%)

✓ Delta features calculated successfully!


---
## 8. Column Cleanup: Remove Useless Columns

In [80]:
print("\n" + "=" * 80)
print("COLUMN CLEANUP: REMOVING USELESS COLUMNS")
print("=" * 80)

# Columns to remove
columns_to_remove = [
    # Empty LogFile timestamp columns (replaced by parsed before/after)
    'lf_creation_time',
    'lf_modified_time',
    'lf_accessed_time',
    'lf_mft_modified_time',
    # Single-value columns (no variance)
    'usn_source_info',  # 100% "Normal"
    'lf_redo',  # 100% "Update Resident Value"
    # Empty column
    'usn_carving_flag'  # 100% empty
]

print(f"\n📋 Columns to remove ({len(columns_to_remove)}):")
for i, col in enumerate(columns_to_remove, 1):
    print(f"  {i}. {col}")

# Remove columns
df_clean = df_parsed.drop(columns=columns_to_remove)

print(f"\n📊 Cleanup Results:")
print(f"  Columns before: {len(df_parsed.columns)}")
print(f"  Columns removed: {len(columns_to_remove)}")
print(f"  Columns after: {len(df_clean.columns)}")
print(f"  Records: {len(df_clean):,} (unchanged)")

# Verify timestomped events preserved
timestomped_before = (df_parsed['is_timestomped'] == 1).sum()
timestomped_after = (df_clean['is_timestomped'] == 1).sum()
print(f"\n✓ Timestomped events:")
print(f"  Before: {timestomped_before}")
print(f"  After: {timestomped_after}")
print(f"  Match: {'✅ YES' if timestomped_before == timestomped_after else '❌ NO'}")


COLUMN CLEANUP: REMOVING USELESS COLUMNS

📋 Columns to remove (7):
  1. lf_creation_time
  2. lf_modified_time
  3. lf_accessed_time
  4. lf_mft_modified_time
  5. usn_source_info
  6. lf_redo
  7. usn_carving_flag

📊 Cleanup Results:
  Columns before: 45
  Columns removed: 7
  Columns after: 38
  Records: 154,550 (unchanged)

✓ Timestomped events:
  Before: 252
  After: 252
  Match: ✅ YES


---
## 9. Column Reorganization: Move case_id to First Position

In [81]:
print("\n" + "=" * 80)
print("COLUMN REORGANIZATION & ROW SORTING")
print("=" * 80)

# Get current column order
current_cols = df_clean.columns.tolist()

print("\n📋 Current column order (first 10):")
for i, col in enumerate(current_cols[:10], 1):
    print(f"  {i:2d}. {col}")

# Move case_id to first position
# Reorder: case_id, eventtime, eventtime_dt, then everything else
priority_cols = ['case_id', 'eventtime', 'eventtime_dt']
other_cols = [col for col in current_cols if col not in priority_cols]
new_order = priority_cols + other_cols

df_clean = df_clean[new_order]

print("\n📋 New column order (first 10):")
for i, col in enumerate(df_clean.columns[:10], 1):
    print(f"  {i:2d}. {col}")

print("\n✓ case_id moved to first position!")

# Sort rows by case_id (1, 2, 3, ..., 12), then by eventtime_dt
print("\n📊 Sorting rows by case_id and eventtime...")
print(f"  Before sort - First case_id: {df_clean['case_id'].iloc[0]}")

df_clean = df_clean.sort_values(by=['case_id', 'eventtime_dt'], ascending=[True, True])
df_clean = df_clean.reset_index(drop=True)

print(f"  After sort - First case_id: {df_clean['case_id'].iloc[0]}")
print(f"  Case order: {sorted(df_clean['case_id'].unique())}")
print("\n✓ Rows sorted by case_id (1→12), then by event time!")


COLUMN REORGANIZATION & ROW SORTING

📋 Current column order (first 10):
   1. lf_lsn
   2. eventtime
   3. lf_event
   4. lf_detail
   5. filename
   6. filepath
   7. lf_target_vcn
   8. lf_cluster_index
   9. case_id
  10. eventtime_dt

📋 New column order (first 10):
   1. case_id
   2. eventtime
   3. eventtime_dt
   4. lf_lsn
   5. lf_event
   6. lf_detail
   7. filename
   8. filepath
   9. lf_target_vcn
  10. lf_cluster_index

✓ case_id moved to first position!

📊 Sorting rows by case_id and eventtime...
  Before sort - First case_id: 10
  After sort - First case_id: 1
  Case order: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

✓ Rows sorted by case_id (1→12), then by event time!


---
## 10. Final Column List

In [82]:
print("\n" + "=" * 80)
print(f"FINAL COLUMN LIST ({len(df_clean.columns)} columns)")
print("=" * 80)

# Categorize columns
final_cols = {
    'Metadata': ['case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key'],
    'LogFile Original': ['lf_lsn', 'lf_event', 'lf_detail', 'lf_target_vcn', 'lf_cluster_index'],
    'LogFile Parsed Timestamps': [
        'lf_creation_time_before', 'lf_creation_time_after',
        'lf_modified_time_before', 'lf_modified_time_after',
        'lf_accessed_time_before', 'lf_accessed_time_after',
        'lf_mft_modified_time_before', 'lf_mft_modified_time_after'
    ],
    'LogFile Delta Features': [
        'creation_time_delta_days', 'creation_time_changed_to_past',
        'modified_time_delta_days', 'modified_time_changed_to_past',
        'accessed_time_delta_days', 'accessed_time_changed_to_past',
        'mft_modified_time_delta_days', 'mft_modified_time_changed_to_past'
    ],
    'LogFile Indicators': ['zero_in_nanoseconds', 'copied_from_file'],
    'UsnJrnl': [
        'usn_usn', 'usn_event_info', 'usn_file_attribute',
        'usn_file_reference_number', 'usn_parent_file_reference_number'
    ],
    'Analysis': ['source', 'time_diff_seconds', 'is_tunneling'],
    'Target': ['is_timestomped']
}

print("\n📋 Columns by category:")
for category, cols in final_cols.items():
    print(f"\n{category} ({len(cols)} columns):")
    for col in cols:
        if col in df_clean.columns:
            print(f"  ✓ {col}")
        else:
            print(f"  ✗ {col} (MISSING!)")

# Count total
expected_cols = sum(len(cols) for cols in final_cols.values())
actual_cols = len(df_clean.columns)

print(f"\n📊 Column Count:")
print(f"  Expected: {expected_cols}")
print(f"  Actual: {actual_cols}")
print(f"  Match: {'✅ YES' if expected_cols == actual_cols else '❌ NO'}")


FINAL COLUMN LIST (38 columns)

📋 Columns by category:

Metadata (6 columns):
  ✓ case_id
  ✓ eventtime
  ✓ eventtime_dt
  ✓ filename
  ✓ filepath
  ✓ merge_key

LogFile Original (5 columns):
  ✓ lf_lsn
  ✓ lf_event
  ✓ lf_detail
  ✓ lf_target_vcn
  ✓ lf_cluster_index

LogFile Parsed Timestamps (8 columns):
  ✓ lf_creation_time_before
  ✓ lf_creation_time_after
  ✓ lf_modified_time_before
  ✓ lf_modified_time_after
  ✓ lf_accessed_time_before
  ✓ lf_accessed_time_after
  ✓ lf_mft_modified_time_before
  ✓ lf_mft_modified_time_after

LogFile Delta Features (8 columns):
  ✓ creation_time_delta_days
  ✓ creation_time_changed_to_past
  ✓ modified_time_delta_days
  ✓ modified_time_changed_to_past
  ✓ accessed_time_delta_days
  ✓ accessed_time_changed_to_past
  ✓ mft_modified_time_delta_days
  ✓ mft_modified_time_changed_to_past

LogFile Indicators (2 columns):
  ✓ zero_in_nanoseconds
  ✓ copied_from_file

UsnJrnl (5 columns):
  ✓ usn_usn
  ✓ usn_event_info
  ✓ usn_file_attribute
  ✓ usn_fil

---
## 11. Save Clean Dataset

In [83]:
print("\n" + "=" * 80)
print("SAVING CLEAN DATASET")
print("=" * 80)

# Save only master dataset (no individual case files)
output_file = OUTPUT_DIR / 'all_cases_combined_clean.csv'
df_clean.to_csv(output_file, index=False, encoding='utf-8-sig')
file_size = output_file.stat().st_size / (1024 * 1024)

print(f"\n✓ Saved clean master dataset:")
print(f"  File: {output_file.name}")
print(f"  Path: {output_file}")
print(f"  Size: {file_size:.2f} MB")
print(f"  Records: {len(df_clean):,}")
print(f"  Columns: {len(df_clean.columns)}")
print(f"  Timestomped: {(df_clean['is_timestomped'] == 1).sum()}")
print(f"\n  Note: Individual case files NOT generated (use case_id column to filter)")


SAVING CLEAN DATASET

✓ Saved clean master dataset:
  File: all_cases_combined_clean.csv
  Path: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1B - Column Cleanup/all_cases_combined_clean.csv
  Size: 76.17 MB
  Records: 154,550
  Columns: 38
  Timestomped: 252

  Note: Individual case files NOT generated (use case_id column to filter)


---
## 12. Summary Report

In [84]:
print("\n" + "=" * 80)
print("PHASE 1B SUMMARY REPORT")
print("=" * 80)

print("\n✅ ANALYSIS & TRANSFORMATION COMPLETE")

print("\n📊 Key Findings:")
print("\n1. LogFile Events:")
print("   - Only Time Reversal events exist (no 'Update' events)")
print("   - All events are explicit timestamp manipulation indicators")

print("\n2. Empty LogFile Timestamp Columns:")
print("   - lf_creation_time, lf_modified_time, etc. are EMPTY (expected)")
print("   - Timestamp details stored in lf_detail text field")
print("   - ✓ Successfully parsed lf_detail to structured columns")

print("\n3. Single-Value Columns (No Variance):")
print("   - usn_source_info: 100% 'Normal' → REMOVED")
print("   - lf_redo: 100% 'Update Resident Value' → REMOVED")
print("   - usn_carving_flag: 100% empty → REMOVED")

print("\n📊 Transformation Results:")
print(f"  Columns removed: 7")
print(f"  Columns added (parsed): 18")
print(f"    - Timestamp before/after: 8 pairs")
print(f"    - Delta features: 8")
print(f"    - Boolean indicators: 2")
print(f"  Net change: +11 columns")
print(f"  Final column count: {len(df_clean.columns)}")
print(f"  Records: {len(df_clean):,} (unchanged)")
print(f"  Timestomped events: {(df_clean['is_timestomped'] == 1).sum()} (unchanged)")

print("\n📊 Column Organization:")
print(f"  ✓ case_id moved to first position")
print(f"  ✓ Logical grouping: Metadata → LogFile → UsnJrnl → Analysis → Target")

print("\n📁 Output Files:")
print(f"  Master dataset: all_cases_combined_clean.csv ({file_size:.2f} MB)")
print(f"  Location: {OUTPUT_DIR}")
print(f"  Individual case files: Not generated (use case_id column to filter)")

print("\n" + "=" * 80)
print("✅ PHASE 1B COMPLETE - READY FOR PHASE 2 (FEATURE ENGINEERING)")
print("=" * 80)


PHASE 1B SUMMARY REPORT

✅ ANALYSIS & TRANSFORMATION COMPLETE

📊 Key Findings:

1. LogFile Events:
   - Only Time Reversal events exist (no 'Update' events)
   - All events are explicit timestamp manipulation indicators

2. Empty LogFile Timestamp Columns:
   - lf_creation_time, lf_modified_time, etc. are EMPTY (expected)
   - Timestamp details stored in lf_detail text field
   - ✓ Successfully parsed lf_detail to structured columns

3. Single-Value Columns (No Variance):
   - usn_source_info: 100% 'Normal' → REMOVED
   - lf_redo: 100% 'Update Resident Value' → REMOVED
   - usn_carving_flag: 100% empty → REMOVED

📊 Transformation Results:
  Columns removed: 7
  Columns added (parsed): 18
    - Timestamp before/after: 8 pairs
    - Delta features: 8
    - Boolean indicators: 2
  Net change: +11 columns
  Final column count: 38
  Records: 154,550 (unchanged)
  Timestomped events: 252 (unchanged)

📊 Column Organization:
  ✓ case_id moved to first position
  ✓ Logical grouping: Metadata →

---
## Next: Phase 2 - Feature Engineering

### Phase 1B Deliverables:
- ✅ **Clean master dataset**: `all_cases_combined_clean.csv` (154,550 records, 38 columns)
- ✅ **All 252 timestomped events preserved**
- ✅ **7 useless columns removed**
- ✅ **18 new columns created** from `lf_detail` parsing
- ✅ **case_id moved to first position** for better readability
- ✅ **Structured timestamp data** ready for ML

### New Features Available for Phase 2:

#### Timestamp Features (Structured):
- Before/after timestamps for CreationTime, ModifiedTime, AccessedTime, MFTModifiedTime
- Delta calculations (in days)
- Direction indicators (changed to past vs future)

#### Manipulation Indicators:
- `zero_in_nanoseconds`: Tool-based manipulation indicator
- `copied_from_file`: SetMACE pattern indicator

### Phase 2 Tasks:

#### 1. Additional Temporal Features
- Event time vs file timestamp discrepancy
- Impossible timestamp sequences
- Timestamp ordering violations
- Z-score of timestamp anomaly

#### 2. Cross-Artifact Features
- Source confidence encoding (both=2, single=1)
- Time difference consistency
- Cross-artifact timestamp agreement

#### 3. Behavioral Features
- Event frequency per file
- Temporal clustering
- File system tunneling patterns
- Event type patterns

#### 4. File-Level Features
- File path depth
- File extension
- System/user/temp folder indicators
- File attribute patterns

**Target**: 60-80 features for ML training

---